# C3 - Day 10: Model 3 evaluation and live demo
Compares three readers on the **128 unseen test documents** (topics the model never saw):

| Reader | What it is |
|---|---|
| **Rule reader** | Fixed rules: headings, a "Plan:" line, TODO placeholders (no learning) |
| **Base Qwen** | Qwen2.5-1.5B-Instruct with the same instructions, not fine-tuned |
| **Fine-tuned (Model 3)** | Qwen2.5-1.5B-Instruct + our LoRA adapter |

Checks: valid JSON, right section, last words copied exactly from the document, right list of remaining sections, right source
label, search phrases that name the open section and the topic. Results are split by kind of document, because rules are
expected to do well on documents with a plan or TODO headings and badly on documents without headings.

Needs `qwen_c3_docs_lora` in `MyDrive/C3/` (notebook 09). **T4 GPU.**

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found. Use Runtime -> Change runtime type -> T4 GPU.'
!pip -q install peft bitsandbytes accelerate

In [ ]:
import os, json, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/C3'
GEN_DIR = os.path.join(BASE, 'data', 'generated')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
ADAPTER_DIR = os.path.join(BASE, 'qwen_c3_docs_lora')
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

def read_jsonl(name):
    with open(os.path.join(GEN_DIR, name), encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = read_jsonl('doc_progress_test.jsonl')
print(len(test_rows), 'test documents')

## Project code (the same files the local tests run)

In [ ]:
"""Synthetic data for Model 3: reading an unfinished text document and saying where the student stopped,
what is still missing and what to search for.

The model never writes any part of the document. Its answer is a short JSON record:
  stopped_at       the section the student was in and the last words written (copied from the document)
  remaining        sections still to do, each with where the information comes from
                   ("your plan", "your headings" or "usual structure for a <document type>")
  search_queries   one web-search phrase per open section (the app turns them into links; nothing is generated)

Run from the repository root:
    python -m components.c3_checkpointing.model3.doc_data
"""

import argparse
import json
import random
import re
from pathlib import Path

SEED = 7
COUNTS = {"train": 640, "val": 64, "test": 128}

DOC_TYPES = {
    "lab_report": {"label": "Lab report", "phrase": "lab report", "sections": ["Introduction", "Aim", "Method", "Results", "Discussion", "Conclusion", "References"]},
    "project_proposal": {"label": "Project proposal", "phrase": "project proposal", "sections": ["Introduction", "Background", "Objectives", "Method", "Risks", "Conclusion", "References"]},
    "essay": {"label": "Essay", "phrase": "essay", "sections": ["Introduction", "Background", "Analysis", "Discussion", "Conclusion", "References"]},
    "requirements_doc": {"label": "Requirements document", "phrase": "software requirements document", "sections": ["Introduction", "Scope", "Requirements", "Design", "Testing", "Risks", "Summary"]},
    "study_notes": {"label": "Study notes", "phrase": "set of study notes", "sections": ["Summary", "Key Points", "Examples", "Questions", "References"]},
}

TOPICS = {
    "train": ["sorting algorithms", "database normalisation", "network latency", "password hashing", "user authentication", "web accessibility",
              "cloud storage", "machine learning basics", "version control", "memory management", "http caching", "unit testing",
              "binary search trees", "operating system scheduling", "api design", "data visualisation", "mobile app performance",
              "sql injection", "agile planning", "image compression"],
    "val": ["recursion", "encryption keys", "responsive layouts", "message queues"],
    "test": ["graph traversal", "dns resolution", "load balancing", "code review", "relational joins", "regular expressions", "docker containers", "event driven design"],
}

SENT = {
    "Introduction": ["This {kind} looks at {topic}.", "{Topic} matters because many systems depend on it.", "The main question here is how {topic} works in practice.", "First, a short overview of {topic} is given."],
    "Aim": ["The aim is to measure how {topic} behaves under different settings.", "We want to show the effect of {topic} with a small experiment.", "The second aim is to compare two ways of handling {topic}."],
    "Background": ["{Topic} has been studied for many years.", "Earlier work describes two common approaches to {topic}.", "The course notes explain the basic terms used for {topic}.", "Most textbooks start from a simple model of {topic}."],
    "Objectives": ["The first objective is to describe how {topic} is used today.", "The second objective is to build a small prototype that uses {topic}.", "The last objective is to check the prototype with a few users."],
    "Scope": ["This document covers {topic} for the first release only.", "Reporting and admin screens are not part of the first release.", "The system will run in a normal web browser."],
    "Method": ["The experiment was run on a laptop with the default settings.", "Each test about {topic} was repeated five times.", "The results were written to a csv file after every run.", "The code for {topic} was kept in one repository."],
    "Requirements": ["The system must let a user log in with a password.", "The system must store the data about {topic} safely.", "Every page must load in under three seconds.", "Users must be able to export their data."],
    "Design": ["The design uses three layers: interface, logic and storage.", "A class diagram for {topic} is shown in figure 1.", "The storage layer keeps one table for each main concept."],
    "Analysis": ["One argument is that {topic} is easier to learn than it looks.", "The evidence about {topic} points in two directions.", "A second point is that the cost of {topic} grows with the size of the system."],
    "Results": ["The first run of {topic} took twelve seconds.", "After the change the same run took only four seconds.", "Table 1 lists all the measurements for {topic}.", "The error rate stayed below two percent."],
    "Discussion": ["The numbers suggest that {topic} has a large effect.", "One limit is that only a small sample was tested.", "A different setup for {topic} might give other results."],
    "Testing": ["Unit tests were written for the main functions of {topic}.", "A short user test was planned with five students.", "Failed tests are listed in the appendix."],
    "Risks": ["The biggest risk is that {topic} takes longer than planned.", "A second risk is that the data may not be available in time.", "Each risk has a person who watches it."],
    "Key Points": ["{Topic} has a simple core idea.", "The most important term to remember is the main rule of {topic}.", "There are two common mistakes with {topic}."],
    "Examples": ["Example one shows {topic} in a very small program.", "Example two uses {topic} on real data.", "The lecturer's example about {topic} is on slide twelve."],
    "Questions": ["Why does {topic} become slow for large inputs?", "When should {topic} not be used?", "How can {topic} be tested?"],
    "Summary": ["In short, {topic} is a central part of the system.", "The main points about {topic} are listed above."],
    "Conclusion": ["In conclusion, {topic} worked as expected.", "The main finding is that {topic} improves the result.", "More work on {topic} is needed in the future."],
    "References": ["Smith, J. (2019). Introduction to {topic}. Pearson.", "Lecture notes, week 4, on {topic}.", "Brown, A. (2021). Practical {topic}. O'Reilly."],
}


SYNONYMS = {
    "Method": ["Methodology", "Procedure", "Method and materials"], "Results": ["Findings", "Observations"],
    "Conclusion": ["Conclusions", "Final remarks"], "Background": ["Literature review"], "Risks": ["Risk assessment"],
    "Testing": ["Test plan"], "Objectives": ["Goals"], "Discussion": ["Analysis of results"], "Requirements": ["Functional requirements"],
}


def _fill(template: str, topic: str, kind: str) -> str:
    return template.format(topic=topic, Topic=topic[0].upper() + topic[1:], kind=kind)


def _heading(name: str, idx: int, style: str) -> str:
    if style == "markdown":
        return f"## {name}"
    if style == "numbered":
        return f"{idx + 1}. {name}"
    if style == "bare":
        return name  # a line with just the heading words, the way many students write
    return name.upper()  # caps


def _tokens(s: str) -> list[str]:
    return re.findall(r"[a-z0-9']+", s.lower())


def build_document(rng: random.Random, split: str) -> dict:
    dtype = rng.choice(list(DOC_TYPES))
    spec = DOC_TYPES[dtype]
    topic = rng.choice(TOPICS[split])
    title = f"{topic[0].upper() + topic[1:]}: {spec['label']}"
    style = rng.choice(["markdown", "numbered", "caps", "bare", "bare"])
    variant = rng.choices(["outline", "placeholders", "plain", "headingless"], [0.25, 0.20, 0.30, 0.25])[0]
    names = spec["sections"]
    shown = [rng.choice(SYNONYMS[n]) if (variant == "plain" and n in SYNONYMS and rng.random() < 0.6) else n for n in names]
    full = [[_fill(t, topic, spec["phrase"]) for t in rng.sample(SENT[n], k=min(len(SENT[n]), rng.randint(2, 4)))] for n in names]

    # where the student stopped: section k, with s sentences written (s == len means the section is finished)
    k = rng.randrange(len(names))
    s = rng.randint(1, len(full[k]))
    written = [list(sec) for sec in full[:k]] + [full[k][:s]]
    partial = s < len(full[k])
    cut_mid = partial and rng.random() < 0.3
    if cut_mid:
        words = written[-1][-1].rstrip(".").split()
        written[-1][-1] = " ".join(words[: max(2, int(len(words) * 0.6))])

    lines = [f"# {title}" if style == "markdown" else title, ""]
    if variant == "outline":
        lines += ["Plan: " + ", ".join(names), ""]
    for i, sec in enumerate(written):
        if variant != "headingless":
            lines += [_heading(shown[i], i, style)]
        lines += [" ".join(sec), ""]
    if variant == "placeholders":
        for i in range(len(written), len(names)):
            lines += [_heading(names[i], i, style), "TODO", ""]
    text = "\n".join(lines).strip() + "\n"

    cur = shown[k] if variant != "headingless" else names[k]
    body_words = " ".join(written[-1]).split()
    last_words = " ".join(body_words[-10:])
    ends_mid = cut_mid
    later = names[k + 1:]
    if variant == "outline":
        src = "your plan"
    elif variant == "placeholders":
        src = "your headings"
    else:
        src = f"usual structure for a {spec['phrase']}"
    remaining = [{"item": n, "from": src} for n in later]
    queries = [f"how to write the {n.lower()} section of a {spec['phrase']} about {topic}" for n in ([names[k]] + later)[:3]]
    answer = {"stopped_at": {"section": cur, "last_words": last_words, "ends_mid_sentence": ends_mid},
              "remaining": remaining, "search_queries": queries}
    return {"split": split, "doc_type": dtype, "topic": topic, "variant": variant, "style": style, "document": text, "answer": answer}


SYSTEM_PROMPT = (
    "You read an unfinished text document that a student was writing before an interruption. "
    "Answer with one JSON object and nothing else, with the keys stopped_at, remaining and search_queries. "
    "stopped_at has the section the student was in, the last words written (copied exactly from the document) "
    "and whether the last sentence is cut off. remaining lists the sections still to do, each with where it comes from: "
    "'your plan', 'your headings' or 'usual structure for a <document type>'. "
    "search_queries has one web-search phrase for each open section. "
    "Never write any part of the document, never give advice, never add facts."
)


def render_input(document: str) -> str:
    return "Document:\n" + document


def to_row(item: dict, idx: int) -> dict:
    user = render_input(item["document"])
    out = json.dumps(item["answer"], ensure_ascii=False)
    return {"id": f"{item['split']}-{idx:04d}", **item, "input": user, "output": out,
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user},
                         {"role": "assistant", "content": out}], "synthetic": True}


def generate_split(split: str) -> list[dict]:
    rng = random.Random(SEED + {"train": 0, "val": 1, "test": 2}[split])
    rows, seen = [], set()
    while len(rows) < COUNTS[split]:
        item = build_document(rng, split)
        if item["document"] in seen:
            continue
        seen.add(item["document"])
        rows.append(to_row(item, len(rows)))
    return rows


"""Rule-based reference reader and evaluation metrics for Model 3 (document progress).

The rule reader is the honest baseline: it uses only headings, a "Plan:" line and TODO placeholders. The learned model
has to beat it where rules are weak (documents with no plan, other heading styles) and must stay valid and grounded.
"""

import json
import re


_LABEL_TO_TYPE = {v["label"].lower(): k for k, v in DOC_TYPES.items()}


def _norm(s: str) -> str:
    return " ".join(re.findall(r"[a-z0-9']+", s.lower()))


def parse_json(text: str) -> dict | None:
    """The first JSON object in the model answer, or None."""
    try:
        start = text.index("{")
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except (ValueError, json.JSONDecodeError):
        return None


_KNOWN = {n.lower() for spec in DOC_TYPES.values() for n in spec["sections"]} | {w.lower() for ws in SYNONYMS.values() for w in ws}


def _heading_name(line: str) -> str | None:
    m = re.match(r"^(?:#{1,3}\s+|\d+\.\s+)(.+)$", line.strip())
    if m:
        return m.group(1).strip()
    t = line.strip()
    if t.lower() in _KNOWN:  # a bare line that is a well-known section name (the rules know every name the data uses)
        return t
    if t and t != "TODO" and t.isupper() and len(t.split()) <= 3 and re.fullmatch(r"[A-Z ]+", t):
        return t.title()
    return None


def rule_reader(document: str) -> dict:
    lines = document.splitlines()
    title = lines[0].lstrip("# ").strip() if lines else ""
    label = title.split(":")[-1].strip().lower()
    topic = title.split(":")[0].strip().lower()
    dtype = _LABEL_TO_TYPE.get(label)
    plan = None
    sections: list[tuple[str, list[str]]] = []
    for ln in lines[1:]:
        if ln.startswith("Plan:"):
            plan = [p.strip() for p in ln[5:].split(",") if p.strip()]
            continue
        name = _heading_name(ln)
        if name:
            sections.append((name, []))
        elif sections and ln.strip():
            sections[-1][1].append(ln.strip())
    written = [(n, b) for n, b in sections if b and b != ["TODO"]]
    placeholders = [n for n, b in sections if b == ["TODO"]]
    cur_name, cur_body = written[-1] if written else ("", [])
    body = " ".join(cur_body)
    last_words = " ".join(body.split()[-10:])
    full = DOC_TYPES[dtype]["sections"] if dtype else []
    ends_mid = bool(body) and not body.rstrip().endswith((".", "?", "!"))
    names = plan or full
    idx = names.index(cur_name) if cur_name in names else -1
    later = names[idx + 1:] if idx >= 0 else []
    if placeholders:
        later, src = placeholders, "your headings"
    elif plan:
        src = "your plan"
    else:
        src = f"usual structure for a {DOC_TYPES[dtype]['phrase']}" if dtype else "usual structure"
    phrase = DOC_TYPES[dtype]["phrase"] if dtype else "document"
    targets = [cur_name] + later
    return {"stopped_at": {"section": cur_name, "last_words": last_words, "ends_mid_sentence": ends_mid},
            "remaining": [{"item": n, "from": src} for n in later],
            "search_queries": [f"how to write the {n.lower()} section of a {phrase} about {topic}" for n in targets[:3]]}


def _prf(pred: set, gold: set) -> tuple[float, float, float]:
    if not pred and not gold:
        return 1.0, 1.0, 1.0
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def score_example(output: str, row: dict) -> dict:
    obj = parse_json(output)
    gold = row["answer"]
    doc_norm = _norm(row["document"])
    res = {"valid_json": obj is not None, "section_ok": False, "last_words_exact": False, "midsentence_ok": False,
           "remaining_f1": 0.0, "source_ok": False, "queries_ok": False, "grounded": False, "queries_count_ok": False}
    if obj is None:
        return res
    st = obj.get("stopped_at") or {}
    rem = obj.get("remaining") or []
    qs = obj.get("search_queries") or []
    res["section_ok"] = _norm(str(st.get("section", ""))) == _norm(gold["stopped_at"]["section"])
    lw = _norm(str(st.get("last_words", "")))
    res["last_words_exact"] = bool(lw) and lw == _norm(gold["stopped_at"]["last_words"])
    res["grounded"] = bool(lw) and lw in doc_norm  # copied from the document, nothing invented
    res["midsentence_ok"] = bool(st.get("ends_mid_sentence")) == gold["stopped_at"]["ends_mid_sentence"]
    pred_items = [_norm(str(r.get("item", ""))) for r in rem if isinstance(r, dict)]
    gold_items = [_norm(r["item"]) for r in gold["remaining"]]
    res["remaining_f1"] = _prf(set(pred_items), set(gold_items))[2]
    gold_src = {r["from"] for r in gold["remaining"]}
    pred_src = {str(r.get("from", "")) for r in rem if isinstance(r, dict)}
    res["source_ok"] = pred_src == gold_src
    res["queries_count_ok"] = len(qs) == len(gold["search_queries"])
    topic = _norm(row["topic"])
    targets = [_norm(r) for r in pred_items] + [_norm(st.get("section", ""))]
    res["queries_ok"] = all(any(t and t[:5] in _norm(str(q)) for t in targets) and topic in _norm(str(q)) for q in qs) and (len(qs) > 0 or not gold["search_queries"])
    return res


def summarise(scores: list[dict]) -> dict:
    keys = list(scores[0])
    return {k: round(100 * sum(float(s[k]) for s in scores) / len(scores), 1) for k in keys}


In [ ]:
"""LoRA training and generation helpers for Model 2.

This file is used twice: the Colab notebooks include its source, and the local tests run the same code
on a tiny random model, so a mistake here is caught before any GPU time is spent.
"""
import inspect
import time

import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def encode_row(tok, row, max_len=1024):
    """Tokenise one chat example; only the assistant answer is trained on (the prompt is masked with -100)."""
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(row["messages"], tokenize=False)
    if not full.startswith(prompt):
        raise ValueError("chat template does not start with the prompt for " + row.get("id", "?"))
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    n = min(len(full_ids), max_len)
    return {"input_ids": full_ids[:n], "attention_mask": [1] * n, "labels": labels[:n]}


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def attach_lora(model, quantized, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none", task_type="CAUSAL_LM",
                        target_modules=LORA_TARGETS)
    return get_peft_model(model, config)


def training_arguments(output_dir, epochs, lr, batch_size, accum, has_eval, fp16, seed):
    kwargs = dict(
        output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, gradient_accumulation_steps=accum, learning_rate=lr,
        lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5, save_strategy="no", report_to="none",
        fp16=fp16, seed=seed, remove_unused_columns=False,
    )
    if has_eval:
        accepted = inspect.signature(TrainingArguments.__init__).parameters
        kwargs["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def train(model, tok, train_rows, val_rows, out_dir, epochs=5, lr=2e-4, batch_size=4, accum=4, fp16=True,
          max_len=1024, seed=42):
    set_seed(seed)
    train_items = [encode_row(tok, r, max_len) for r in train_rows]
    val_items = [encode_row(tok, r, max_len) for r in val_rows]
    collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
    args = training_arguments(out_dir, epochs, lr, batch_size, accum, bool(val_items), fp16, seed)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_items),
                      eval_dataset=ListDataset(val_items) if val_items else None, data_collator=collator)
    model.config.use_cache = False
    start = time.time()
    result = trainer.train()
    model.config.use_cache = True
    info = {
        "seconds": round(time.time() - start, 1), "train_loss": float(result.training_loss),
        "epochs": epochs, "learning_rate": lr, "batch_size": batch_size, "grad_accumulation": accum,
        "max_len": max_len, "seed": seed, "n_train": len(train_items), "n_val": len(val_items),
        "longest_example_tokens": max(len(i["input_ids"]) for i in train_items),
        "history": trainer.state.log_history,
    }
    return trainer, info


@torch.no_grad()
def generate_batch(model, tok, message_lists, max_new_tokens=300, batch_size=8):
    """Greedy generation for a list of chat prompts (system + user messages); returns the answers."""
    model.eval()
    tok.padding_side = "left"
    answers = []
    for i in range(0, len(message_lists), batch_size):
        chunk = message_lists[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            new_tokens = out[j][enc["input_ids"].shape[1]:]
            answers.append(tok.decode(new_tokens, skip_special_tokens=True).strip())
    return answers


## Load the base model and the adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
print('Loaded base model + adapter')

## Write the answers (greedy decoding, about 10 to 15 minutes)

In [ ]:
prompts = [r['messages'][:2] for r in test_rows]
outputs = {'Rule reader': [json.dumps(rule_reader(r['document'])) for r in test_rows]}
outputs['Fine-tuned'] = generate_batch(model, tok, prompts, max_new_tokens=500, batch_size=8)
with model.disable_adapter():
    outputs['Base Qwen'] = generate_batch(model, tok, prompts, max_new_tokens=500, batch_size=8)
print({k: len(v) for k, v in outputs.items()})

## Score every answer

In [ ]:
rows = []
for system, outs in outputs.items():
    for r, o in zip(test_rows, outs):
        rows.append({'system': system, 'variant': r['variant'], **score_example(o, r)})
scores = pd.DataFrame(rows)
cols = ['valid_json', 'section_ok', 'last_words_exact', 'grounded', 'remaining_f1', 'source_ok', 'queries_ok', 'midsentence_ok']
order = ['Rule reader', 'Base Qwen', 'Fine-tuned']
overall = (scores.groupby('system')[cols].mean() * 100).round(1).loc[order]
print('All 128 test documents (%):'); print(overall.to_string())
for v in ['outline', 'placeholders', 'plain', 'headingless']:
    sub = scores[scores.variant == v]
    print(f"\n{v} ({int(len(sub) / 3)} documents):"); print((sub.groupby('system')[cols].mean() * 100).round(1).loc[order].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
kinds = ['outline', 'placeholders', 'plain', 'headingless']
w = 0.26
for i, s in enumerate(order):
    vals = [100 * scores[(scores.system == s) & (scores.variant == k)][['section_ok', 'remaining_f1']].mean().mean() for k in kinds]
    ax.bar(np.arange(len(kinds)) + (i - 1) * w, vals, w, label=s)
ax.set_xticks(np.arange(len(kinds))); ax.set_xticklabels(['With a plan line', 'With TODO headings', 'Plain (other heading words)', 'No headings at all'])
ax.set_ylabel('% correct (section and remaining list)'); ax.set_ylim(0, 105); ax.set_title('Model 3: reading an unfinished document'); ax.legend(loc='lower left')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'model3_evaluation.png'), dpi=150); plt.show()
overall.to_csv(os.path.join(OUTPUT_DIR, 'results_model3_evaluation.csv'))
scores.to_csv(os.path.join(OUTPUT_DIR, 'results_model3_per_example.csv'), index=False)

## One example per kind of document, three readers

In [ ]:
for v in ['plain', 'headingless']:
    i = next(k for k, r in enumerate(test_rows) if r['variant'] == v)
    print('=' * 90); print(v); print(test_rows[i]['document'])
    for s in order:
        print(f'--- {s} ---'); print(outputs[s][i][:600])

## LIVE DEMO: paste your own unfinished document
Replace the text in the form (or paste a document into `document`) and press Run. The model reads it, says where you stopped,
what is still missing and gives search links. It writes nothing for your document.

In [ ]:
#@title Your unfinished document { run: "auto" }
document = """My project report: Project proposal

Introduction
This proposal is about a small tool that helps students restart their work after an interruption.
Background
Earlier studies show that people need extra time to return to a task. The main finding is that a short reminder of the goal helps.
Method
We will build a prototype and ask five students to try it. The prototype keeps the goal, the last file and the""" #@param {type:"string"}
import urllib.parse, time
t0 = time.time()
msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': render_input(document.strip() + '\n')}]
raw = generate_batch(model, tok, [msgs], max_new_tokens=500, batch_size=1)[0]
obj = parse_json(raw)
print(f'Model answer in {time.time() - t0:.1f} s'); print(raw)
if obj:
    st = obj.get('stopped_at', {})
    print('\nYou stopped in:', st.get('section')); print('Last words:', st.get('last_words'))
    print('Still to do:', ', '.join(f"{r['item']} ({r['from']})" for r in obj.get('remaining', [])))
    print('Search links:')
    for q in obj.get('search_queries', []):
        print('  https://www.google.com/search?q=' + urllib.parse.quote_plus(q))
else:
    print('The answer was not valid JSON: the app would ignore it and show no analysis.')